# 🎯 Module 13 · 04 — Bag of Words

> **Stage:** Machine Learning → NLP  
> **Level:** Beginner → Intermediate  
> **Module:** 13 — NLP Basics / 04 — Bag of Words  
> **Date:** 28 January 2026  
> **Previous:** 03 — Text Representation  
> **Next:** 05 — TF-IDF

---

**Bag of Words (BoW)** is one of the simplest and most important techniques for converting text into numerical features.

> **Represent a document by the words it contains and how frequently those words appear.**

In this notebook we'll cover:

- Simple BoW example
- Why it's called a "bag"
- Word frequency
- Vocabulary & feature dimensions
- `CountVectorizer` (scikit-learn)
- Binary BoW
- N-grams with BoW
- Controlling vocabulary (`min_df`, `max_df`, `max_features`)
- Stop words
- Advantages & limitations
- BoW vs TF-IDF
- **Vocabulary leakage** and how to avoid it

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what BoW is and why it's called a **bag**
- Build a BoW matrix from scratch
- Use `CountVectorizer` from scikit-learn
- Inspect vocabularies and sparse matrices
- Use **binary** BoW
- Add **n-grams** to BoW
- Control the vocabulary with `min_df`, `max_df`, `max_features`
- Explain when to remove **stop words**
- Explain the main **limitations** of BoW
- Distinguish **BoW from TF-IDF**
- Avoid **vocabulary leakage**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer

np.random.seed(42)
print('✅ Setup ready.')

In [ ]:
# Sample corpus used throughout the notebook
documents = [
    "I love Python",
    "I love machine learning",
    "Python is powerful",
    "This laptop is good and this laptop is fast",
]

for i, doc in enumerate(documents, 1):
    print(f'Doc {i}: {doc}')

---
## 🧠 1. Simple Example

Two documents:

```text
Document 1:  "I love Python"
Document 2:  "I love machine learning"
```

Vocabulary:

```text
["I", "love", "Python", "machine", "learning"]
```

Representations:

```text
Doc 1: [1, 1, 1, 0, 0]
Doc 2: [1, 1, 0, 1, 1]
```

| Word | Position |
| --- | --- |
| I | 1 |
| love | 2 |
| Python | 3 |
| machine | 4 |
| learning | 5 |

In [ ]:
# Manual BoW on a tiny corpus
tiny_corpus = ["I love Python", "I love machine learning"]

# Build vocabulary
vocab = sorted({w.lower() for doc in tiny_corpus for w in doc.split()})
print('Vocabulary:', vocab)

# BoW vectors
def manual_bow(doc, vocab):
    tokens = doc.lower().split()
    return [tokens.count(w) for w in vocab]

print()
for doc in tiny_corpus:
    print(f'{doc:<30} -> {manual_bow(doc, vocab)}')

---
## 👜 2. Why Is It Called Bag of Words?

Consider:

```text
"I love Python"
"Python love I"
```

A basic BoW representation treats them **similarly** because they contain the same words.

The representation focuses on:

```text
Which words?
+
How many times?
```

rather than:

```text
Exact word order?
```

> BoW is **simple** and **useful**, but it loses word order.

In [ ]:
# Same words, different order → same BoW vector
a = "dog bites man"
b = "man bites dog"

tiny_vocab = sorted(set((a + ' ' + b).split()))
print('Vocab:', tiny_vocab)
print()
print(f'{a!r:<20} -> {manual_bow(a, tiny_vocab)}')
print(f'{b!r:<20} -> {manual_bow(b, tiny_vocab)}')
print()
print('Same vector -> BoW cannot tell them apart.')

---
## 🔢 3. Word Frequency

Example:

```text
"This laptop is good and this laptop is fast"
```

Word counts:

```text
this   → 2
laptop → 2
is     → 2
good   → 1
and    → 1
fast   → 1
```

This is a **count-based** representation.

In [ ]:
from collections import Counter

text = "This laptop is good and this laptop is fast"
counts = Counter(text.lower().split())

print('Word counts:')
for w, c in counts.most_common():
    print(f'  {w:<8} {c}')

---
## 📚 4. Vocabulary

The **vocabulary** is the set of **unique terms** found in the training documents.

```text
Docs:  "I love Python",  "I love AI"
Vocab: ["I", "love", "Python", "AI"]
```

The vocabulary size determines the **number of features**.

If the vocabulary has 10,000 words → each document becomes a 10,000-dim vector.

Most documents use only a small subset → the matrix is **sparse**.

---
## 🛠️ 5. CountVectorizer

Scikit-learn provides `CountVectorizer`:

```python
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)
```

It does two things:

```text
Text
 ↓
Build Vocabulary
 ↓
Count Word Occurrences
```

In [ ]:
# CountVectorizer on the sample corpus
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

print('Vocabulary:')
print(vectorizer.get_feature_names_out())

print('\nMatrix shape:', X.shape)
print('Type        :', type(X).__name__)
print('Non-zero    :', X.nnz, '/', X.shape[0] * X.shape[1])
print(f'Sparsity    : {1 - X.nnz / (X.shape[0] * X.shape[1]):.1%}')

---
## 📖 6. Viewing the Vocabulary

```python
vectorizer.get_feature_names_out()
```

Returns the learned vocabulary in **alphabetical order** — each entry corresponds to a **column** in the BoW matrix.

In [ ]:
vocab = vectorizer.get_feature_names_out()
print(f'Vocabulary size: {len(vocab)}')
print('First 10 tokens:', vocab[:10])

# Map feature name -> column index
print('\nFeature index mapping:')
for i, w in enumerate(vocab[:10]):
    print(f'  {i:>3}  {w}')

---
## 🔍 7. Viewing the Matrix

```python
X.toarray()
```

Structure:

```text
Rows    -> Documents
Columns -> Words
Values  -> Word counts
```

In [ ]:
# Inspect the BoW matrix as a DataFrame
df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f'Doc{i+1}' for i in range(len(documents))],
)

print(df)

# Focus on a few columns to see actual counts
cols = [c for c in ['this', 'laptop', 'is', 'love', 'python'] if c in df.columns]
print('\nSelected counts:')
print(df[cols])

---
## 0️⃣ 8. Binary Bag of Words

Instead of storing counts:

```text
python -> 2
machine -> 1
```

binary BoW becomes:

```text
python -> 1
machine -> 1
```

Useful when **presence** matters more than **frequency**.

```python
CountVectorizer(binary=True)
```

In [ ]:
# Binary BoW on the same corpus
vec_bin = CountVectorizer(binary=True)
X_bin = vec_bin.fit_transform(documents)

df_bin = pd.DataFrame(
    X_bin.toarray(),
    columns=vec_bin.get_feature_names_out(),
    index=[f'Doc{i+1}' for i in range(len(documents))],
)

# Show a few columns where counts mattered in the original
show = [c for c in ['this', 'laptop', 'is'] if c in df_bin.columns]
print('Binary BoW (selected columns):')
print(df_bin[show])

---
## 🔗 9. N-Grams with BoW

Basic BoW uses **unigrams** (single words).

We can also include **bigrams** (two-word phrases):

```python
CountVectorizer(ngram_range=(1, 2))
```

This creates **unigrams + bigrams** together.

### Why it helps

Phrases like:

```text
"not good"
"very good"
"customer service"
```

carry meaning that **individual words miss**.

In [ ]:
# Bigrams with BoW
vec_bigram = CountVectorizer(ngram_range=(1, 2))
X_bi = vec_bigram.fit_transform(documents)

print('Vocabulary size (uni + bi):', len(vec_bigram.get_feature_names_out()))

bigrams = [t for t in vec_bigram.get_feature_names_out() if ' ' in t]
print('\nBigrams found:')
for b in bigrams:
    print(' ', b)

In [ ]:
# N-gram example that captures negation
neg_docs = ["the movie is good", "the movie is not good"]

uni = CountVectorizer(ngram_range=(1, 1)).fit_transform(neg_docs)
bi  = CountVectorizer(ngram_range=(1, 2)).fit_transform(neg_docs)

print('Unigram features:', uni.shape[1])
print('Bigram  features:', bi.shape[1])
print()
print('Bigram vocab contains "not good":', 'not good' in CountVectorizer(ngram_range=(1, 2)).fit(neg_docs).get_feature_names_out())

---
## 🎛️ 10. Controlling the Vocabulary

Large datasets can have huge vocabularies.

### `min_df`
Ignore words appearing in **too few** documents.

```python
CountVectorizer(min_df=2)
```

### `max_df`
Ignore words appearing in **too many** documents (i.e., extremely common words).

```python
CountVectorizer(max_df=0.90)
```

### `max_features`
Limit the vocabulary size.

```python
CountVectorizer(max_features=5000)
```

In [ ]:
# Compare vocab size under different settings
configs = [
    ('default',                       CountVectorizer()),
    ('min_df=1',                      CountVectorizer(min_df=1)),
    ('min_df=2',                      CountVectorizer(min_df=2)),
    ('max_df=0.9',                    CountVectorizer(max_df=0.9)),
    ('max_features=3',                CountVectorizer(max_features=3)),
    ('binary=True',                   CountVectorizer(binary=True)),
    ('ngram_range=(1,2)',             CountVectorizer(ngram_range=(1, 2))),
    ('stop_words="english"',         CountVectorizer(stop_words='english')),
]

print(f'{"Config":<28} {"Vocab":>6} {"Shape":>12}')
print('-' * 50)
for name, v in configs:
    Xc = v.fit_transform(documents)
    print(f'{name:<28} {len(v.get_feature_names_out()):>6} {str(Xc.shape):>12}')

---
## 🚫 11. BoW and Stop Words

```python
CountVectorizer(stop_words='english')
```

Removes common English words like `the`, `is`, `of`.

### ⚠️ But be careful

```text
"I do not like this"
```

Removing **not** can flip the meaning.

> Always evaluate preprocessing choices **against the task**.

In [ ]:
# Compare with and without stop-word removal
text_neg = ["I do not like this product"]

v1 = CountVectorizer()
v2 = CountVectorizer(stop_words='english')

v1.fit(text_neg); v2.fit(text_neg)

print('Without stop removal:', list(v1.get_feature_names_out()))
print('With stop removal   :', list(v2.get_feature_names_out()))
print()
print('Notice how "not" is stripped — dangerous for sentiment analysis.')

---
## ✅ 12. Advantages of BoW

- **Simple** — easy to understand and implement
- **Fast** — works efficiently with classical ML
- **Strong baseline** — often surprisingly hard to beat
- **Interpretable** — each feature is a specific word

```text
Feature 15 → "excellent"
Feature 92 → "terrible"
```

This makes model behavior **easy to inspect**.

---
## ⚠️ 13. Limitations of BoW

### 1. Ignores word order

```text
"dog bites man"
"man bites dog"
```

get similar representations.

### 2. No semantic understanding

`car` and `automobile` are treated as **separate** features.

### 3. High dimensionality

Large vocabularies → thousands or millions of features.

### 4. Sparse matrix

Most documents use only a small fraction of the vocabulary.

### 5. Weak handling of context

The meaning of a word often depends on the surrounding words.

---
## 🤖 14. BoW with Machine Learning

Classical NLP architecture:

```text
Text
 ↓
Preprocessing
 ↓
CountVectorizer
 ↓
Bag of Words
 ↓
Logistic Regression
 ↓
Prediction
```

Example:

```text
"I love this product"
        ↓
[0, 1, 0, 1, 0, ...]
        ↓
Classifier
        ↓
Positive
```

In [ ]:
# Tiny end-to-end example: BoW + Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import random

# Build a small synthetic sentiment corpus
random.seed(42)
positive = [
    "I love this product", "amazing quality", "absolutely wonderful",
    "best purchase ever", "highly recommend", "fantastic experience",
    "great service", "excellent quality",
]
negative = [
    "worst product ever", "terrible quality", "absolutely awful",
    "waste of money", "do not recommend", "horrible experience",
    "bad service", "poor quality",
]

texts = positive + negative
labels = [1] * len(positive) + [0] * len(negative)

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

vec = CountVectorizer()
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr, y_train)

preds = clf.predict(Xte)
print('Test accuracy:', round(accuracy_score(y_test, preds), 3))

# Predict a new sentence
new = ["I love this amazing product"]
print('New prediction:', clf.predict(vec.transform(new))[0], '(1=positive, 0=negative)')

---
## ⚖️ 15. BoW vs TF-IDF

| Feature | BoW | TF-IDF |
| --- | --- | --- |
| Counts words | ✅ Yes | ⚖️ Weighted values |
| Uses vocabulary | ✅ | ✅ |
| Sparse | ✅ | ✅ |
| Considers word frequency | ✅ | ✅ |
| Penalizes common words | ❌ No | ✅ Yes |
| Simple | Very | Relatively |
| Common use | Baselines | Classification / Search |

### The key difference

```text
BoW      -> How many times does the word appear?
TF-IDF   -> How important is the word to this document
            compared with the rest of the corpus?
```

---
## 🧪 16. Practical Example

```python
from sklearn.feature_extraction.text import CountVectorizer

texts = [
    "I love machine learning",
    "I love Python",
    "Python is powerful",
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)
```

Workflow:

```text
fit_transform()
      ↓
Learn vocabulary
+
Transform documents
      ↓
Numerical matrix
```

### ⚠️ For **new** text

```python
X_new = vectorizer.transform(new_text)
```

Use `transform()` — **not** `fit_transform()` — because the vocabulary must come from the training data.

In [ ]:
# Demonstrate transform() on new text
texts = ["I love machine learning", "I love Python", "Python is powerful"]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

new_text = ["I love Python"]
X_new = vectorizer.transform(new_text)

print('Training matrix shape:', X.shape)
print('New matrix shape     :', X_new.shape)
print('Same vocabulary      :', list(vectorizer.get_feature_names_out()))
print('New doc vector       :', X_new.toarray().tolist())

---
## 🚫 17. Avoiding Vocabulary Leakage

### ❌ Incorrect

```text
Entire Dataset
      ↓
Fit Vectorizer
      ↓
Train/Test Split
```

### ✅ Correct

```text
Dataset
   ↓
Train/Test Split
   ↓
Training Data → Fit Vectorizer
   ↓
Testing Data  → Transform Only
```

> The test set must **never** influence the vocabulary.  
> Use a scikit-learn **Pipeline** to enforce this automatically.

In [ ]:
# The correct way with a Pipeline
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

pipe = Pipeline([
    ('bow', CountVectorizer()),
    ('clf', LogisticRegression(max_iter=1000)),
])

scores = cross_val_score(pipe, texts * 4, ([1, 0, 1] * 4), cv=3)
print('Pipeline CV scores:', scores.round(3))
print('Mean             :', scores.mean().round(3))
print()
print('Pipeline ensures the vectorizer only sees the training fold in each CV split.')

---
## 🧠 18. Mental Model

```text
Documents
   ↓
Unique Words
   ↓
Vocabulary
   ↓
Count Occurrences
   ↓
Sparse Matrix
   ↓
Machine Learning
```

### 🧭 Golden Rule

> **Bag of Words converts documents into numerical vectors based on word occurrence or frequency, without trying to understand the deeper meaning of the text.**

---
## 🏋️ 19. Hands-On Exercises

### Exercise 1 — Manual BoW
For the corpus:
```python
["red apple", "green apple", "red car"]
```
build the vocabulary and BoW matrix **by hand**.  
Then verify with `CountVectorizer`.

### Exercise 2 — Binary vs Counts
Use the corpus from Exercise 1 with `binary=True`.  
Which columns change values?

### Exercise 3 — N-grams
Apply `CountVectorizer(ngram_range=(1, 2))` to the corpus:
```python
["the movie is not good", "the movie is very good"]
```
What bigrams appear? Which one captures negation?

### Exercise 4 — `min_df` sweep
Try `min_df ∈ {1, 2, 3}` on a bigger corpus.  
How does the vocabulary size change?

### Exercise 5 — Compare BoW vs TF-IDF
Train a Logistic Regression on the same dataset with:
- `CountVectorizer`
- `TfidfVectorizer`

Compare F1 scores.

### Exercise 6 — Vocabulary leakage
Write two code snippets:
1. Fit `CountVectorizer` **before** the split (**wrong**)
2. Fit **after** the split, or inside a Pipeline (**correct**)

Which gives a more honest evaluation?

---
## 🎤 20. Interview Questions

1. **What is Bag of Words?**  
   A representation of a document as a vector of word counts / occurrences.

2. **Why is it called a "bag"?**  
   Because word order is discarded — only which words and how often matter.

3. **What is a vocabulary in BoW?**  
   The set of unique tokens found in the training documents.

4. **What does each column represent?**  
   One vocabulary word.

5. **What does each row represent?**  
   One document.

6. **What is a sparse matrix?**  
   A matrix where most values are zero — typical for BoW.

7. **How does `CountVectorizer` work?**  
   It builds a vocabulary and counts word occurrences for each document.

8. **What does `binary=True` do?**  
   Records only presence (1) or absence (0) of each word.

9. **Why use n-grams?**  
   They capture short phrases such as `"not good"`.

10. **What do `min_df` / `max_df` / `max_features` do?**  
    They filter or limit the vocabulary.

11. **When should you NOT remove stop words?**  
    When negation matters (e.g., sentiment) or when common words carry meaning.

12. **Main limitations of BoW?**  
    No word order, no semantic understanding, high dimensionality, sparse.

13. **BoW vs TF-IDF?**  
    BoW counts words; TF-IDF weights them by document-vs-corpus importance.

14. **What is vocabulary leakage?**  
    Fitting the vectorizer on test data as well — inflates performance.

15. **How do you prevent it?**  
    Split first, fit only on training data, or use a scikit-learn `Pipeline`.

---
## 🏁 21. Key Takeaways

### You now understand

- What Bag of Words is
- Why it is called a **bag**
- Vocabulary as a feature space
- Word counts as numerical features
- Sparse matrices
- `CountVectorizer`
- Binary BoW
- N-grams within BoW
- `min_df`, `max_df`, `max_features`
- Advantages and limitations
- BoW vs TF-IDF
- How BoW connects to ML models
- **Vocabulary leakage** and how to prevent it

### The progression

```text
Text
 ↓
Vocabulary
 ↓
Word Counts
 ↓
Sparse Matrix
 ↓
ML Model
 ↓
Prediction
```

---

### 🔗 What's Next?

**05 — TF-IDF**

We'll move from **simple word counts → measuring how important words are within a document collection**.